<a href="https://colab.research.google.com/github/Semiaris0404/SentimentAnalysisNLP/blob/main/nlp_sentiment_Finalproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## System Setup
Install dependencies and verify GPU availability. Remeber to ensure **Runtime GPU** before running.

In [ ]:
!pip install datasets transformers accelerate evaluate scikit-learn huggingface_hub -q

In [ ]:
import torch, numpy as np, pandas as pd, evaluate, time, re
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, TrainingArguments, Trainer)
from datasets import load_dataset

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")  #'cuda'

## Data Loading & Subsampling
Load IMDB from HuggingFace. Reserve a fixed 15% validation split, then create balanced subsets for the data-efficiency sweep. Subsampling is done **once here** and reused across all models.

In [ ]:
imdb = load_dataset("imdb")

# Fixed val split
split           = imdb["train"].train_test_split(test_size=0.15, seed=SEED)
imdb_train_pool = split["train"]   # ~21,250 — all subsampling draws from here
imdb_val        = split["test"]    # ~3,750  — fixed validation for every run
imdb_test       = imdb["test"]     # 25,000  — held-out final evaluation

print(f"Train pool: {len(imdb_train_pool)} | Val: {len(imdb_val)} | Test: {len(imdb_test)}")

In [ ]:
def subsample_balanced(dataset, n, seed=SEED):
    """Return a class-balanced subsample of n examples."""
    rng  = np.random.default_rng(seed)
    pos  = [i for i, ex in enumerate(dataset) if ex["label"] == 1]
    neg  = [i for i, ex in enumerate(dataset) if ex["label"] == 0]
    half = n // 2
    chosen = rng.choice(pos, half, replace=False).tolist() + \
             rng.choice(neg, half, replace=False).tolist()
    rng.shuffle(chosen)
    return dataset.select(chosen)

sweep_sizes = [100, 500, 1000, 5000, len(imdb_train_pool)]
subsets = {n: subsample_balanced(imdb_train_pool, n) for n in sweep_sizes[:-1]}
subsets[len(imdb_train_pool)] = imdb_train_pool

print({k: len(v) for k, v in subsets.items()})

##Encoder Fine-Tuning
Fine-tune three encoder models across all five data sizes. Evaluation metric: macro-F1.  
Epoch count is scaled by training size (10 for n≤500, 5 for n=1000, 3 for n≥5000).  
Each run saves a `preds_<model>_<n>.csv` file for error analysis in Section 5.

In [ ]:
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1":       f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
    }

In [ ]:
def run_one(model_name, n, results_log):
    """Fine-tune model_name on subset of size n. Appends result row and saves prediction CSV."""
    print(f"\n{'='*55}\n{model_name}  |  n={n}\n{'='*55}")
    short = model_name.split("/")[-1]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tok(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

    train_tok = subsets[n].map(tok, batched=True)
    val_tok   = imdb_val.map(tok, batched=True)
    test_tok  = imdb_test.map(tok, batched=True)
    for ds in [train_tok, val_tok, test_tok]:
        ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    model  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    epochs = 10 if n <= 500 else (5 if n <= 1000 else 3)

    args = TrainingArguments(
        output_dir                  = f"./ckpt/{short}_{n}",
        num_train_epochs            = epochs,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        learning_rate               = 2e-5,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        logging_steps               = 50,
        seed                        = SEED,
        fp16                        = (device == "cuda"),
        report_to                   = "none",
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=train_tok, eval_dataset=val_tok,
                      compute_metrics=compute_metrics)

    t0 = time.time(); trainer.train();              train_t = time.time() - t0
    t1 = time.time(); out = trainer.predict(test_tok); infer_t = time.time() - t1
    m  = compute_metrics((out.predictions, out.label_ids))
    print(f"  ✓  F1={m['f1']:.4f}  |  Acc={m['accuracy']:.4f}")

    results_log.append({
        "model": short, "n_train": n,
        "test_f1": round(m["f1"], 4), "test_accuracy": round(m["accuracy"], 4),
        "train_time_s": round(train_t, 1), "inference_time_s": round(infer_t, 1),
        "n_params_M": sum(p.numel() for p in model.parameters()) // 1_000_000,
    })

    preds = np.argmax(out.predictions, axis=-1)
    pd.DataFrame({
        "text":    [ex["text"] for ex in imdb_test],
        "label":   out.label_ids,
        "pred":    preds,
        "correct": (preds == out.label_ids),
    }).to_csv(f"preds_{short}_{n}.csv", index=False)
    print(f"  Saved: preds_{short}_{n}.csv")

    del model; torch.cuda.empty_cache()

In [ ]:
#DistilBERT (66M params — fastest, run first)
distilbert_results = []
for n in sweep_sizes:
    run_one("distilbert-base-uncased", n, distilbert_results)

df_d = pd.DataFrame(distilbert_results)
df_d.to_csv("results_distilbert.csv", index=False)
print(df_d[["n_train","test_f1","test_accuracy"]].to_string(index=False))

In [ ]:
#BERT-base (109M params)
bert_results = []
for n in sweep_sizes:
    run_one("bert-base-uncased", n, bert_results)

df_b = pd.DataFrame(bert_results)
df_b.to_csv("results_bert.csv", index=False)
print(df_b[["n_train","test_f1","test_accuracy"]].to_string(index=False))

In [ ]:
# RoBERTa-base (124M params)
roberta_results = []
for n in sweep_sizes:
    run_one("roberta-base", n, roberta_results)

df_r = pd.DataFrame(roberta_results)
df_r.to_csv("results_roberta.csv", index=False)
print(df_r[["n_train","test_f1","test_accuracy"]].to_string(index=False))

In [ ]:
df_all = pd.concat([
    pd.read_csv("results_distilbert.csv"),
    pd.read_csv("results_bert.csv"),
    pd.read_csv("results_roberta.csv"),
], ignore_index=True)
df_all.to_csv("encoder_sweep_results_ALL.csv", index=False)

pivot = df_all.pivot(index="n_train", columns="model", values="test_f1")
pivot.columns.name = None
print("\n── Test F1 by model and training size ──")
print(pivot.to_string())

##LLaMA Prompting (Zero-Shot & 4-Shot)
No weight updates, inference only. LLaMA-3.1-8B-Instruct is prompted with a system instruction to output exactly one word: *Positive* or *Negative*.  
Results are saved every 500 examples as a checkpoint against session disconnects.

In [ ]:
from huggingface_hub import login
login()  #token, have access to meta-llama/Llama-3.1-8B-Instruct

In [ ]:
MODEL_ID  = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer_llama = AutoTokenizer.from_pretrained(MODEL_ID)
llama_model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
llama_model.eval()
print("LLaMA loaded")

In [ ]:
SYSTEM_MSG = ("You are a sentiment classifier. "
              "Given a movie review, reply with exactly one word: "
              "Positive or Negative. No explanation.")

FEW_SHOT_EXAMPLES = [
    {"review": "An absolute masterpiece. The performances were breathtaking.", "label": "Positive"},
    {"review": "I walked out after twenty minutes. Painfully boring.",          "label": "Negative"},
    {"review": "Gorgeous cinematography and a haunting soundtrack. One of the best this year.", "label": "Positive"},
    {"review": "A complete waste of time. The plot made no sense.",             "label": "Negative"},
]

def make_zero_shot_prompt(review):
    return [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": f"Review: {review[:1500]}\nSentiment:"}]

def make_four_shot_prompt(review):
    messages = [{"role": "system", "content": SYSTEM_MSG}]
    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user",      "content": f"Review: {ex['review']}\nSentiment:"})
        messages.append({"role": "assistant", "content": ex["label"]})
    messages.append({"role": "user", "content": f"Review: {review[:1500]}\nSentiment:"})
    return messages

In [ ]:
def classify_review(messages, max_new_tokens=5):
    encoded        = tokenizer_llama.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    input_ids      = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        output = llama_model.generate(
            input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer_llama.eos_token_id,
        )
    generated = tokenizer_llama.decode(
        output[0][input_ids.shape[-1]:], skip_special_tokens=True
    ).strip().lower()

    if "positive" in generated: return 1
    if "negative" in generated: return 0
    return -1  #parse failure


def run_llama_condition(condition_name, prompt_fn, test_dataset, save_every=500):
    results, parse_failures = [], 0
    t0 = time.time()

    for i, ex in enumerate(test_dataset):
        pred = classify_review(prompt_fn(ex["text"]))
        if pred == -1:
            parse_failures += 1; pred = 0
        results.append({"text": ex["text"], "label": ex["label"],
                         "pred": pred, "correct": int(pred == ex["label"])})

        if (i + 1) % save_every == 0:
            elapsed = time.time() - t0
            eta     = elapsed / (i + 1) * (len(test_dataset) - i - 1)
            print(f"  [{i+1}/{len(test_dataset)}] elapsed {elapsed/60:.1f}m | "
                  f"ETA {eta/60:.1f}m | parse fails: {parse_failures}")
            pd.DataFrame(results).to_csv(f"preds_llama_{condition_name}_checkpoint.csv", index=False)

    df = pd.DataFrame(results)
    df.to_csv(f"preds_llama_{condition_name}.csv", index=False)
    print(f"\n {condition_name} done | parse failures: {parse_failures}/{len(test_dataset)}")
    return df

In [ ]:
print("Starting zero-shot...")
df_zero = run_llama_condition("zeroshot", make_zero_shot_prompt, imdb_test)

print("\nStarting 4-shot...")
df_four = run_llama_condition("fourshot", make_four_shot_prompt, imdb_test)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def llama_metrics(df, condition_name):
    labels, preds = df["label"].values, df["pred"].values
    return {
        "model": "llama-3.1-8b-instruct", "n_train": condition_name,
        "test_f1":       round(f1_score(labels, preds, average="macro"), 4),
        "test_accuracy": round(accuracy_score(labels, preds), 4),
        "train_time_s": 0, "inference_time_s": None, "n_params_M": 8000,
    }

df_llama = pd.DataFrame([llama_metrics(df_zero, "zeroshot"),
                          llama_metrics(df_four, "fourshot")])
df_llama.to_csv("results_llama.csv", index=False)
print(df_llama[["n_train","test_f1","test_accuracy"]].to_string(index=False))


##Error Analysis
Stratified analysis of misclassifications across four linguistic categories: **negation**, **sarcasm**, **mixed sentiment**, and **domain vocabulary**.  
Focus regime: n=500 (most divergent low-resource point). Cross-model overlap reveals which phenomena are paradigm-specific vs. universal.

In [ ]:
FOCUS_N = 500
models_short = {
    "distilbert": "distilbert-base-uncased",
    "bert":       "bert-base-uncased",
    "roberta":    "roberta-base",
}
preds = {}
for short, full in models_short.items():
    preds[short] = pd.read_csv(f"preds_{full}_{FOCUS_N}.csv")
    print(f"{short} n={FOCUS_N}: {(~preds[short]['correct'].astype(bool)).sum()} errors")

preds["llama_zero"] = pd.read_csv("preds_llama_zeroshot.csv")
preds["llama_four"] = pd.read_csv("preds_llama_fourshot.csv")
for name in ["llama_zero", "llama_four"]:
    print(f"{name}: {(~preds[name]['correct'].astype(bool)).sum()} errors")

In [ ]:
def tag_primary_error(text):
    """Assign one primary linguistic error category. Priority: negation > sarcasm > mixed > domain > other."""
    t = text.lower()
    if re.search(r"\b(not|never|no one|nobody|nothing|n't|neither|nor|hardly|barely|lacks?|fails?)\b", t):
        return "negation"
    if (re.search(r"\b(oh great|yeah right|totally|absolutely|brilliant|masterpiece|flawless)\b", t) and
        re.search(r"\b(terrible|awful|worst|boring|stupid|waste|horrible)\b", t)) or \
       text.count("!") > 3 or text.count("...") > 2:
        return "sarcasm"
    if re.search(r"\b(but|however|although|though|despite|while|whereas|yet|still|nevertheless)\b", t):
        return "mixed_sentiment"
    if re.search(r"\b(cinematography|screenplay|director|sequel|franchise|plot twist|"
                 r"character development|third act|score|soundtrack|pacing|ensemble)\b", t):
        return "domain_vocab"
    return "other"

In [ ]:
CATEGORIES  = ["negation", "sarcasm", "mixed_sentiment", "domain_vocab", "other"]
SAMPLE_SIZE = 20

def stratified_error_sample(df, model_name, n=SAMPLE_SIZE):
    errors = df[~df["correct"].astype(bool)].copy()
    errors["error_type"] = errors["text"].apply(tag_primary_error)
    rows = []
    for cat in CATEGORIES:
        pool    = errors[errors["error_type"] == cat]
        k       = min(n, len(pool))
        if k == 0:
            print(f"  [{model_name}] WARNING: no examples for '{cat}'"); continue
        sampled = pool.sample(n=k, random_state=42).copy()
        sampled["model"] = model_name; sampled["category"] = cat
        rows.append(sampled)
        print(f"  [{model_name}] {cat:20s}: {len(pool):4d} total → sampled {k}")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

print("="*55)
all_samples = {}
for name, df in preds.items():
    print(f"\n{name}")
    all_samples[name] = stratified_error_sample(df, name)

error_df = pd.concat(all_samples.values(), ignore_index=True)
error_df[["model","category","text","label","pred"]].to_csv("error_analysis_stratified.csv", index=False)
print(f"\nSaved error_analysis_stratified.csv — {len(error_df)} rows")

In [ ]:
def error_profile(df, model_name):
    errors = df[~df["correct"].astype(bool)].copy()
    errors["error_type"] = errors["text"].apply(tag_primary_error)
    total = len(errors)
    row   = {"model": model_name, "total_errors": total,
             "error_rate": round(total / len(df), 4)}
    for cat in CATEGORIES:
        row[f"{cat}_pct"] = round((errors["error_type"] == cat).sum() / total * 100, 1) if total else 0
    return row

profile_df = pd.DataFrame([error_profile(df, name) for name, df in preds.items()])
profile_df.to_csv("error_profile_summary.csv", index=False)

print("\nError type distribution (% of all errors per model)")
print("="*70)
cols = ["model","error_rate"] + [f"{c}_pct" for c in CATEGORIES]
print(profile_df[cols].to_string(index=False))

In [ ]:
# Cross-model overlap: DistilBERT n=500 vs LLaMA zero-shot
df_enc = preds["distilbert"].copy()
df_llm = preds["llama_zero"].copy()
df_enc["error_type"] = df_enc["text"].apply(tag_primary_error)

enc_fail_llm_win = df_enc[(~df_enc["correct"].astype(bool)) &  df_llm["correct"].astype(bool)].copy()
llm_fail_enc_win = df_enc[( df_enc["correct"].astype(bool)) & ~df_llm["correct"].astype(bool)].copy()
llm_fail_enc_win["error_type"] = llm_fail_enc_win["text"].apply(tag_primary_error)

print(f"DistilBERT fails, LLaMA wins: {len(enc_fail_llm_win)}")
print(enc_fail_llm_win["error_type"].value_counts().to_string())
print(f"\nLLaMA fails, DistilBERT wins: {len(llm_fail_enc_win)}")
print(llm_fail_enc_win["error_type"].value_counts().to_string())

enc_fail_llm_win.to_csv("overlap_enc_fail_llm_win.csv", index=False)
llm_fail_enc_win.to_csv("overlap_llm_fail_enc_win.csv", index=False)

In [ ]:
def show_examples(df, category, n=3):
    """Print n readable misclassification examples for a given error category."""
    subset = df[df["error_type"] == category].head(n)
    for _, row in subset.iterrows():
        true_lbl = "POS" if row["label"] == 1 else "NEG"
        pred_lbl = "POS" if row["pred"]  == 1 else "NEG"
        print(f"  True: {true_lbl} | Pred: {pred_lbl}")
        print(f"  {row['text'][:220]}...\n")

print("~ DistilBERT negation failures (LLaMA got these right)~")
show_examples(enc_fail_llm_win, "negation", n=3)

print("~LLaMA zero-shot failures (DistilBERT got these right)~")
show_examples(llm_fail_enc_win, "domain_vocab", n=3)